# LRS3 Preprocessing (Clean)

This notebook keeps only the LRS3 preprocessing steps actually used:
- Use existing LRS3 video/audio as-is (no fps/audio conversion)
- Use existing landmark files
- Crop stabilized mouth ROI (96x96)

It is separated from TCD-TIMIT for clarity.

In [ ]:
from pathlib import Path
import os
import subprocess

# ===== User config =====
LRS3_ROOT = Path(r"E:\lrs3_rj\lrs3")
SPLIT = "trainval"  
SPEAKERS = None      

FFMPEG_EXE = "ffmpeg"
PYTHON_EXE = os.environ.get("PYTHON_EXECUTABLE", "python")

ALIGN_MOUTH_PY = Path(r"C:\Users\irish\Computer_Electronic_Engineering_Year5\AVSR_project\av_hubert\av_hubert\avhubert\preparation\align_mouth_stabilised.py")
MEAN_FACE_NPY = Path(r"C:\Users\irish\Computer_Electronic_Engineering_Year5\AVSR_project\assets\mean_face\20words_mean_face.npy")

VIDEO_DIR = LRS3_ROOT / SPLIT
LMK_DIR = LRS3_ROOT / "landmark" / SPLIT
ROI_DIR = LRS3_ROOT / "roi" / SPLIT

assert VIDEO_DIR.exists(), f"Missing split dir: {VIDEO_DIR}"
assert LMK_DIR.exists(), f"Missing landmark dir: {LMK_DIR}"
assert ALIGN_MOUTH_PY.exists(), f"Missing crop script: {ALIGN_MOUTH_PY}"
assert MEAN_FACE_NPY.exists(), f"Missing mean face: {MEAN_FACE_NPY}"
print("Config loaded.")

In [ ]:
def run_cmd(cmd: list):
    cmd = [str(c) for c in cmd]
    print(">>", " ".join(cmd))
    subprocess.run(cmd, check=True)

def get_speakers(base: Path, only):
    all_spk = sorted([p.name for p in base.iterdir() if p.is_dir()])
    return all_spk if only is None else [s for s in only if (base / s).exists()]

def iter_stems(spk_dir: Path):
    return sorted([p.stem for p in spk_dir.glob("*.mp4")])


In [ ]:
# 96x96 stabilized mouth cropping
INPUT_FOR_CROP = VIDEO_DIR
speakers = get_speakers(INPUT_FOR_CROP, SPEAKERS)

for spk in speakers:
    vid_spk = INPUT_FOR_CROP / spk
    lmk_spk = LMK_DIR / spk
    out_spk = ROI_DIR / spk
    out_spk.mkdir(parents=True, exist_ok=True)

    stems = sorted([p.stem for p in vid_spk.glob("*.mp4") if (lmk_spk / f"{p.stem}.pkl").exists()])
    if not stems:
        print(f"{spk}: no matched video/landmark pairs, skipping crop")
        continue

    filelist = out_spk / "file.list"
    filelist.write_text("\n".join(stems) + "\n", encoding="utf-8")

    run_cmd([
        PYTHON_EXE, ALIGN_MOUTH_PY,
        "--video-direc", vid_spk,
        "--landmark-direc", lmk_spk,
        "--filename-path", filelist,
        "--save-direc", out_spk,
        "--mean-face", MEAN_FACE_NPY,
        "--crop-width", "96",
        "--crop-height", "96",
        "--start-idx", "48",
        "--stop-idx", "68",
        "--window-margin", "12",
        "--ffmpeg", FFMPEG_EXE,
        "--rank", "0",
        "--nshard", "1",
        "--gray", "0",
        "--stab-alpha", "0.15",
        "--mouth-alpha", "0.20",
        "--max-t-jump", "3.0",
        "--max-r-jump", "3.0",
        "--max-s-jump", "0.03",
    ])

print("Done: LRS3 mouth ROI crops")

## Notes
- This notebook is LRS3-only by design.
- LRS3 preprocessing here assumes video/audio formats and landmark files already exist.
- Set SPEAKERS to a small list for debugging.